In [1]:
import pandas as pd

file_path = "../data/raw/default of credit card clients.xls"

df = pd.read_excel(file_path, header=1)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 25)


In [2]:
target = "default payment next month"

X = df.drop(columns=["ID", target])
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Feature columns:")
print(X.columns.tolist())

Features shape: (30000, 23)
Target shape: (30000,)
Feature columns:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [3]:
print("Categorical features:")
print(["SEX", "EDUCATION", "MARRIAGE"])

print("\nRepayment status features:")
print(["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"])

print("\nNumeric features:")
print([
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
])

Categorical features:
['SEX', 'EDUCATION', 'MARRIAGE']

Repayment status features:
['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

Numeric features:
['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [4]:
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"{col}:")
    print(sorted(X[col].unique()))
    print()

SEX:
[np.int64(1), np.int64(2)]

EDUCATION:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

MARRIAGE:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]



In [5]:
print("Missing values in features:")
print(X.isnull().sum().sum())

print("\nMissing values in target:")
print(y.isnull().sum())

Missing values in features:
0

Missing values in target:
0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (24000, 23)
X_test: (6000, 23)
y_train: (24000,)
y_test: (6000,)


In [7]:
print("Training set:")
print(y_train.value_counts(normalize=True).mul(100))

print("\nTest set:")
print(y_test.value_counts(normalize=True).mul(100))

Training set:
default payment next month
0    77.879167
1    22.120833
Name: proportion, dtype: float64

Test set:
default payment next month
0    77.883333
1    22.116667
Name: proportion, dtype: float64


In [8]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numeric_features = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

repayment_features = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

print("Categorical:", len(categorical_features))
print("Numeric:", len(numeric_features))
print("Repayment:", len(repayment_features))

Categorical: 3
Numeric: 14
Repayment: 6


In [9]:
print("Dataset:", df.shape)
print("Features:", X.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Preprocessor exists:", "preprocessor" in globals())

Dataset: (30000, 25)
Features: (30000, 23)
Training set: (24000, 23)
Test set: (6000, 23)
Preprocessor exists: False


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (24000, 33)
Processed test shape: (6000, 33)


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]
)

print("Logistic Regression pipeline created successfully.")

Logistic Regression pipeline created successfully.


In [17]:
model.fit(X_train, y_train)

print("Model training completed successfully.")

Model training completed successfully.


In [18]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Default", "Default"]
    )
)

Accuracy: 67.82 %

Confusion Matrix:
[[3230 1443]
 [ 488  839]]

Classification Report:
              precision    recall  f1-score   support

  No Default       0.87      0.69      0.77      4673
     Default       0.37      0.63      0.46      1327

    accuracy                           0.68      6000
   macro avg       0.62      0.66      0.62      6000
weighted avg       0.76      0.68      0.70      6000



In [15]:

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

# 2. Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 3. Display detailed evaluation metrics
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Default", "Default"]
    )
)


Accuracy: 80.88 %

Confusion Matrix:
[[4529  144]
 [1003  324]]

Classification Report:
              precision    recall  f1-score   support

  No Default       0.82      0.97      0.89      4673
     Default       0.69      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.76      0.61      0.62      6000
weighted avg       0.79      0.81      0.77      6000



In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [20]:
random_forest_model.fit(X_train, y_train)

print("Random Forest training completed successfully.")

Random Forest training completed successfully.


In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Generate predictions
y_pred_rf = random_forest_model.predict(X_test)

# Calculate accuracy
print(
    "Random Forest Accuracy:",
    round(accuracy_score(y_test, y_pred_rf) * 100, 2),
    "%"
)

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Display classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=["No Default", "Default"]
    )
)

Random Forest Accuracy: 80.15 %

Confusion Matrix:
[[4170  503]
 [ 688  639]]

Classification Report:
              precision    recall  f1-score   support

  No Default       0.86      0.89      0.88      4673
     Default       0.56      0.48      0.52      1327

    accuracy                           0.80      6000
   macro avg       0.71      0.69      0.70      6000
weighted avg       0.79      0.80      0.80      6000



In [22]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Default Precision": precision_score(y_test, y_pred, zero_division=0),
        "Default Recall": recall_score(y_test, y_pred, zero_division=0),
        "Default F1": f1_score(y_test, y_pred, zero_division=0)
    },
    {
        "Model": "Logistic Regression Balanced",
        "Accuracy": 0.6782,
        "Default Precision": 0.37,
        "Default Recall": 0.63,
        "Default F1": 0.46
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Default Precision": precision_score(y_test, y_pred_rf, zero_division=0),
        "Default Recall": recall_score(y_test, y_pred_rf, zero_division=0),
        "Default F1": f1_score(y_test, y_pred_rf, zero_division=0)
    }
])

comparison = comparison.round(4)

comparison

,Model,Accuracy,Default Precision,Default Recall,Default F1
0,Logistic Regression,0.6782,0.3677,0.6323,0.4649
1,Logistic Regression Balanced,0.6782,0.3700,0.6300,0.4600
2,Random Forest,0.8015,0.5595,0.4815,0.5176


In [23]:
# Preserve predictions from each trained model

y_pred_lr = model.predict(X_test)
y_pred_rf = random_forest_model.predict(X_test)

print("Logistic Regression predictions saved:", len(y_pred_lr))
print("Random Forest predictions saved:", len(y_pred_rf))

Logistic Regression predictions saved: 6000
Random Forest predictions saved: 6000


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Default Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Default Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "Default F1": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

comparison = pd.DataFrame([
    evaluate_model(
        "Logistic Regression Balanced",
        y_test,
        y_pred_lr
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        y_pred_rf
    )
])

comparison = comparison.round(4)
comparison

,Model,Accuracy,Default Precision,Default Recall,Default F1
0,Logistic Regression Balanced,0.6782,0.3677,0.6323,0.4649
1,Random Forest,0.8015,0.5595,0.4815,0.5176


In [25]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

rf_200 = clone(random_forest_model)

rf_200.set_params(
    classifier__n_estimators=200
)

print("Random Forest with 200 trees is ready.")

Random Forest with 200 trees is ready.


In [26]:
rf_200.fit(X_train, y_train)

print("Random Forest (200 trees) training completed!")

Random Forest (200 trees) training completed!


In [27]:
from sklearn.metrics import accuracy_score, classification_report

# Generate predictions
y_pred_rf200 = rf_200.predict(X_test)

# Calculate accuracy
print(
    "Random Forest (200 trees) Accuracy:",
    round(accuracy_score(y_test, y_pred_rf200) * 100, 2),
    "%"
)

# Evaluate the Default class
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf200,
        target_names=["No Default", "Default"]
    )
)

Random Forest (200 trees) Accuracy: 80.17 %

Classification Report:
              precision    recall  f1-score   support

  No Default       0.86      0.89      0.88      4673
     Default       0.56      0.49      0.52      1327

    accuracy                           0.80      6000
   macro avg       0.71      0.69      0.70      6000
weighted avg       0.79      0.80      0.80      6000



In [28]:
from sklearn.base import clone

rf_depth10 = clone(random_forest_model)

rf_depth10.set_params(
    classifier__n_estimators=100,
    classifier__max_depth=10
)

print("Random Forest with max_depth=10 is ready.")

Random Forest with max_depth=10 is ready.


In [29]:
rf_depth10.fit(X_train, y_train)

print("Random Forest (max_depth=10) training completed!")

Random Forest (max_depth=10) training completed!


In [30]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_depth10 = rf_depth10.predict(X_test)

print(
    "Random Forest (max_depth=10) Accuracy:",
    round(accuracy_score(y_test, y_pred_depth10) * 100, 2),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_depth10,
        target_names=["No Default", "Default"]
    )
)

Random Forest (max_depth=10) Accuracy: 77.6 %

Classification Report:
              precision    recall  f1-score   support

  No Default       0.88      0.83      0.85      4673
     Default       0.49      0.59      0.54      1327

    accuracy                           0.78      6000
   macro avg       0.69      0.71      0.70      6000
weighted avg       0.79      0.78      0.78      6000



In [31]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

print("Cross-validation setup is ready.")

Cross-validation setup is ready.


In [32]:
from sklearn.model_selection import cross_validate

# Models to compare
models = {
    "Logistic Regression": model,
    "Random Forest": random_forest_model,
    "Random Forest (200 Trees)": rf_200,
    "Random Forest (Depth 10)": rf_depth10
}

# Store cross-validation results
cv_results = []

for name, estimator in models.items():
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Default Precision": scores["test_precision"].mean(),
        "Default Recall": scores["test_recall"].mean(),
        "Default F1": scores["test_f1"].mean()
    })

# Display comparison
cv_comparison = pd.DataFrame(cv_results)

cv_comparison = cv_comparison.sort_values(
    by="Default F1",
    ascending=False
).reset_index(drop=True)

cv_comparison.round(4)

,Model,Accuracy,Default Precision,Default Recall,Default F1
0,Random Forest (Depth 10),0.7791,0.5007,0.5924,0.5426
1,Random Forest (200 Trees),0.8038,0.5642,0.4971,0.5285
2,Random Forest,0.8030,0.5619,0.4965,0.5272
3,Logistic Regression,0.6887,0.3810,0.6506,0.4805


In [33]:
cv_comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Cross-validation results saved successfully.")

OSError: Cannot save file into a non-existent directory: '../reports'

In [34]:
from pathlib import Path

report_path = Path("../reports/model_comparison.csv")

print("Notebook directory:", Path.cwd())
print("Report path:", report_path.resolve())
print("Reports directory exists:", report_path.parent.is_dir())

Notebook directory: /Users/raghad/Desktop/FinSight-AI/notebooks
Report path: /Users/raghad/Desktop/FinSight-AI/reports/model_comparison.csv
Reports directory exists: True


In [35]:
cv_comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Cross-validation results saved successfully.")

Cross-validation results saved successfully.


In [36]:
# Select the best candidate from cross-validation
final_model = rf_depth10

# Train on the full training set
final_model.fit(X_train, y_train)

print("Final candidate trained successfully.")

Final candidate trained successfully.


In [37]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on unseen test data
y_pred_final = final_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_final)

print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_final,
        target_names=["No Default", "Default"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

Final Model Accuracy: 77.60%

Classification Report:
              precision    recall  f1-score   support

  No Default       0.88      0.83      0.85      4673
     Default       0.49      0.59      0.54      1327

    accuracy                           0.78      6000
   macro avg       0.69      0.71      0.70      6000
weighted avg       0.79      0.78      0.78      6000


Confusion Matrix:
[[3868  805]
 [ 539  788]]


In [38]:
# Predict the probability of default for each test case
y_proba_final = final_model.predict_proba(X_test)[:, 1]

print("First 10 predicted default probabilities:")
print(y_proba_final[:10])

First 10 predicted default probabilities:
[0.3230656  0.35594913 0.38518022 0.3264699  0.14120965 0.51247563
 0.24885314 0.1455724  0.27265251 0.27054901]


In [39]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Compare two classification thresholds
for threshold in [0.50, 0.40]:
    y_pred_threshold = (
        y_proba_final >= threshold
    ).astype(int)

    print(f"\nThreshold: {threshold:.2f}")
    print(
        "Default Precision:",
        round(precision_score(y_test, y_pred_threshold), 4)
    )
    print(
        "Default Recall:",
        round(recall_score(y_test, y_pred_threshold), 4)
    )
    print(
        "Default F1:",
        round(f1_score(y_test, y_pred_threshold), 4)
    )


Threshold: 0.50
Default Precision: 0.4947
Default Recall: 0.5938
Default F1: 0.5397

Threshold: 0.40
Default Precision: 0.3817
Default Recall: 0.7136
Default F1: 0.4974


In [40]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training subset:", X_train_sub.shape)
print("Validation set:", X_val.shape)
print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).round(4))

Training subset: (19200, 23)
Validation set: (4800, 23)

Validation target distribution:
default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [41]:
from sklearn.base import clone

# Create an independent copy of the selected model
threshold_model = clone(rf_depth10)

# Train using only the training subset
threshold_model.fit(X_train_sub, y_train_sub)

print("Threshold-selection model trained successfully.")

Threshold-selection model trained successfully.


In [42]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Predict default probabilities on validation data
y_val_proba = threshold_model.predict_proba(X_val)[:, 1]

# Compare different thresholds
threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Default Precision": precision_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Default Recall": recall_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Default F1": f1_score(
            y_val, y_val_pred, zero_division=0
        )
    })

threshold_comparison = pd.DataFrame(threshold_results)

threshold_comparison.round(4)

,Threshold,Default Precision,Default Recall,Default F1
0,0.30,0.2965,0.8992,0.4459
1,0.35,0.3363,0.8230,0.4775
2,0.40,0.3993,0.7147,0.5123
3,0.45,0.4497,0.6394,0.5280
4,0.50,0.4964,0.5800,0.5350
5,0.55,0.5375,0.5405,0.5390
6,0.60,0.5725,0.4906,0.5284


In [43]:
best_row = threshold_comparison.loc[
    threshold_comparison["Default F1"].idxmax()
]

selected_threshold = best_row["Threshold"]

print("Selected Threshold:", selected_threshold)
print("Validation Default F1:", round(best_row["Default F1"], 4))

Selected Threshold: 0.55
Validation Default F1: 0.539


In [44]:
from sklearn.base import clone

# Create a fresh copy of the selected model
final_model = clone(rf_depth10)

# Train on the full training set
final_model.fit(X_train, y_train)

print("Final model trained on the full training set.")
print("Selected threshold:", selected_threshold)

Final model trained on the full training set.
Selected threshold: 0.55


In [45]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict default probabilities on the test set
y_proba_final = final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
y_pred_final_055 = (
    y_proba_final >= selected_threshold
).astype(int)

# Evaluate the model
print("Selected Threshold:", selected_threshold)

print(
    "\nAccuracy:",
    round(accuracy_score(y_test, y_pred_final_055), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_final_055,
        target_names=["No Default", "Default"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final_055))

Selected Threshold: 0.55

Accuracy: 0.7945

Classification Report:
              precision    recall  f1-score   support

  No Default       0.87      0.87      0.87      4673
     Default       0.54      0.53      0.53      1327

    accuracy                           0.79      6000
   macro avg       0.70      0.70      0.70      6000
weighted avg       0.79      0.79      0.79      6000


Confusion Matrix:
[[4063  610]
 [ 623  704]]


In [46]:
import joblib

artifact = {
    "model": final_model,
    "threshold": float(selected_threshold),
    "model_name": "Random Forest",
    "max_depth": 10,
    "features": list(X_train.columns),
    "target": "default payment next month"
}

joblib.dump(artifact, "../models/finsight_model.joblib")

print("Model artifact saved successfully.")
print("Threshold:", artifact["threshold"])
print("Features:", len(artifact["features"]))

Model artifact saved successfully.
Threshold: 0.55
Features: 23


In [47]:
import joblib
from pathlib import Path

model_path = Path("../models/finsight_model.joblib")

loaded_artifact = joblib.load(model_path)

loaded_model = loaded_artifact["model"]
loaded_threshold = loaded_artifact["threshold"]

print("Model loaded successfully!")
print("Model type:", type(loaded_model.named_steps["classifier"]).__name__)
print("Threshold:", loaded_threshold)
print("Number of features:", len(loaded_artifact["features"]))
print("File exists:", model_path.exists())

Model loaded successfully!
Model type: RandomForestClassifier
Threshold: 0.55
Number of features: 23
File exists: True
